## Bayesian Spline and Sensitivity Test Code 
##### This code has been adapted from a version produced by Dr. Roger Creel. It importants the cosmogenic nuclide exposure age data from the Informal Cosmogenic Exposure age Database (ICE-D). 

In [2]:
# Use ve3PaleoSTeHM environment

import matplotlib
matplotlib.rcParams.update(matplotlib.rcParamsDefault)

In [3]:
# Load modules
import matplotlib.pyplot as plt # Used for plotting 
import matplotlib
import numpy as np # Used for numerical computations
import pandas as pd # For handling tabular data 
import xarray as xr #For working with labeled multi-dimensional arrays 
import os

# Import statistical and optimization tools
from scipy.stats import norm
from scipy import interpolate
from scipy.interpolate import BSpline # For spline fitting 
from scipy.stats import norm, multivariate_normal
from scipy.optimize import curve_fit
from scipy.io import loadmat # For reading MatLab files 
from scipy.optimize import minimize
from scipy.interpolate import UnivariateSpline
from scipy.odr import ODR, Model, RealData # For orthogonal distance regression 

# Matplotlib configuration for advanced plotting 
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.patches as patches
from matplotlib import cm
from matplotlib import colors as mcolors
import matplotlib as mpl
from matplotlib.gridspec import GridSpec
from matplotlib.lines import Line2D
import matplotlib.gridspec as gridspec
import matplotlib.ticker as mticker

# Importing colormap 
import cmocean.cm as cmo
import cmocean as cmo

# System utilities 
import sys
import time

# Import Cartopy for map projections and geographic visualizations 
import cartopy  # Map projections libary
import cartopy.crs as ccrs  # Projections list
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER

# For geospatial and statistical processing 
import pyproj
import statsmodels.api as sm
import geopandas as gpd
import rioxarray

# Importing polartoolkit
from polartoolkit import fetch, maps, regions

# Linear algebra tools 
from numpy.linalg import lstsq

# Machine learning libraries 
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression
from sklearn.cluster import DBSCAN
from sklearn.utils import resample
from util import * 

# Bootstrap and BSpline functions 
import numpy as np
from astropy.stats import bootstrap
from scipy.interpolate import BSpline

#pyro.set_rng_seed(0) #set random seed used in this notebook

# Setting plotting style 
%matplotlib inline
font = {'weight':'normal',
       'size':22,
       'family':'Helvetica'}
matplotlib.rcParams['xtick.major.size'] = 20
matplotlib.rcParams['ytick.major.size'] = 20
matplotlib.rcParams['axes.labelsize'] = 22
matplotlib.rcParams['figure.figsize'] = (12, 6)
matplotlib.rcParams['legend.frameon'] = 'True'
matplotlib.rc('font',**font)

# Turn off warnings 
import warnings
warnings.filterwarnings("ignore")

# Custom fonts for plots 
f1 = 30
f2 = 24
f3 = 20

# Define utility function for replacing nth occurence of a substring 
def replace_nth(txt, sub,repl,nth):
    arr=txt.split(sub)
    part1=sub.join(arr[:nth])
    part2=sub.join(arr[nth:])
    return part1+repl+part2

# Function to prepare data arrays for plotting and analysis 
def make_data_xy(grp):
    grp['elv_zero_st'] = grp.elv_comp 
    grp['elv_err_st'] = grp.elv_err
    grp = grp.sort_values('age') # Sort by age 
    x = grp.age.values[::-1].astype(float) * -1 # np.round(grp.age * 2, -1) / 2 # Reverse and negate ages
    y = grp.elv_zero_st.values[::-1].astype(float) # Reverse elevations 
    yerr = grp.elv_err_st.values[::-1].astype(float) # Reverse elevation errors 
    xerr = grp.age_err.values[::-1].astype(float) # Reverse age errors 
    return x, y, xerr, yerr

# Function to prepare data arrays for plotting and analysis 
# del plot_uncertainty_boxes, plot_tem_regression
def plot_uncertainty_boxes(x, y, x_error, y_error, ax=None, alpha=1, lw=3,
                           fc=(1,0.82,0.86,0.15), ec=(0.7,0,0,0.7), zorder=2):
    '''
    A function to plot uncertainty box for data with vertical and horizontal uncertainties.

    -----------------Input-------------------
    x: a 1-D array of input data
    y: a 1-D array of ouput data
    x_error: a 1-D array containing 2 sigma uncertainty of input data
    y_error: a 1-D array containing 2 sigma uncertainty of output data

    ---------------Return-------------------
    ax: a matplotlib ax of output plot
    '''
    if ax == None: ax = plt.subplot(111)
    for i in range(len(x)):
        ax.add_patch(plt.Rectangle((x[i] - x_error[i], y[i] - y_error[i]), 2 * x_error[i], 2 *  y_error[i], 
                                    fill=True, fc=fc,ec=ec, linewidth=lw, zorder=zorder, alpha=alpha))

    ax.set_xlabel('Age (BP)')
    ax.set_ylabel('Elevation above \npresent-day ice (m)')
    return ax

# Function to plot temporal regression results 
def plot_tem_regression(data_age, data, data_age_sigma, data_elv_sigma, mean_age, mean_elv,
                        lower_bound, upper_bound, mean_rate, lower_bound_rate, upper_bound_rate,
                        name, color='C0', fig=None, axes=None, yhi=0, xhi=0, LGM_lim=0,
                       placename=''):
    '''
    A function to create matplotlib plot for temporal regression results.

    -----------------Inputs-------------------
    data_age: a 1-D array of rsl age data
    data_rsl: a 1-D array of rsl data
    data_age_sigma: a 1-D array containing 1 sigma uncertainty of rsl age data
    data_rsl_sigma: a 1-D array containing 1 sigma uncertainty of rsl data
    mean_rsl_age: a 1-D array of testing age data, i.e., new_X for GP regression
    mean_rsl: a 1-D array of mean rsl regression results
    rsl_sd: a 1-D array of rsl regression standard deviation
    rsl_rate_age: a 1-D array of testing age data for rsl rate
    rsd_rate: a 1-D array of mean rsl rate regression results
    rate_sd: a 1-D array of rsl rate regression standard deviation
    axes: matplotlib axes, if None, create a new figure
    save: bool, whether to save the plot

    ---------------Output-------------------
    A matplotlib plot of temporal regression results which contains
    three sub-plots: 1) RSL data with uncertainty box and mean regression line; 2) RSL rate
    with uncertainty box and mean regression line; 3) Residual plot.


    '''

    if len(axes) < 1:
        fig, axes = plt.subplots(nrows=1,
                                    ncols=2,
                                    figsize=(36, 12)
                                    )
    # Left output subplot: data with uncertainty boxes 
    ax = axes[0]
    plot_uncertainty_boxes(data_age,
                            data,
                            data_age_sigma * 2,
                            data_elv_sigma * 2,
                            ax=ax,
                           lw=0.3,
                           fc='none',
                           ec='k',
                           alpha=1,
                           zorder=2
                            )
    
    plot_uncertainty_boxes(data_age,
                            data,
                            data_age_sigma * 2,
                            data_elv_sigma * 2,
                            ax=ax,
                           fc=color,
                           ec='k',
                           alpha=0.1,
                           zorder=1
                            )
    # Right output subplot: data with uncertainty boxes
    plot_uncertainty_boxes(data_age,
                            data,
                            data_age_sigma * 2,
                            data_elv_sigma * 2,
                            ax=ax,
                           fc='w',
                           ec='w',
                           alpha=1,
                           zorder=0
                            )

    ax.plot(mean_age,mean_elv,linewidth=3,zorder=11,color='k',)
    ax.plot(mean_age,lower_bound,linewidth=1,zorder=11,color='k')
    ax.plot(mean_age,upper_bound,linewidth=1,zorder=11,color='k')
    ax.fill_between( mean_age, lower_bound, upper_bound, color=color, alpha=0.4, zorder=10, label=placename)
    ax.fill_between(mean_age, lower_bound,upper_bound,color='w',alpha=1, zorder=9)
    ylo = np.minimum(-10, np.min(data) - np.mean(data_elv_sigma) * 2)
    xhi = np.maximum(xhi, np.max(data_age) + np.mean(data_age_sigma) * 2 + 300)
    ax.set_ylim(ylo, yhi)
    ax.set_xlim(0, xhi)
    # ax.set_xlim(0, 20000)
    ax.axhline(0, color='k', lw=1)
    ax.grid(alpha=0.3)

    # PLot the LGM limit-assigned in 'icedcurrent_v14_EM.xlsx'
    if LGM_lim > 0:
        ax.axhline(LGM_lim, color=color, ls='--', label=f'{placename} Local LGM Limit')
    
    
    ####### RATE PLOT ##########
    ax = axes[1]

    ax.plot(mean_age,
            mean_rate,
            linewidth=3,
            color=color,
            )
    
    ax.fill_between(mean_age,
                    lower_bound_rate,
                    upper_bound_rate,
                    linewidth=3,
                    label=placename,
                    color=color,
                    alpha=0.3,
                )

    # Get xlim from previous plots in case those are already set
    _, yhi_rate = ax.get_ylim()
    _, xhi_rate = ax.get_xlim()

    yhi_rate = np.maximum(yhi_rate, np.nanmax(upper_bound_rate) + 5)
    # yhi_rate = np.minimum(yhi_rate, 200)

    ### ADD DUMMY LINE FOR LEGEND ###
    if LGM_lim > 0:
        lgmname = placename.split(':')[0]
        ax.axhline(-100, color=color, ls='--', label=f'{lgmname} Local LGM Limit')

    ax.grid(alpha=0.3)
    ax.set_xlim(0, xhi)
    # ax.set_xlim(0, 20000)

    ax.set_ylim(-5, yhi_rate)
    ax.axhline(0, color='k', lw=1)
    
    ax.set_xlabel('Age (ka)')
    ax.set_ylabel('Rate of ice thinning (m/kyr)')
    ax.legend(loc='best')

    # Add titles and return figure 
    fig.suptitle(f'Site {name}')
    return fig, axes


print('done')

done


## Available Sensitivity Tests ##

In [4]:
# | **Test** | **Effect** | **Where Used** | **Notes** |
# |----------|------------|----------------|-----------|
# | **C1b**  | Use weighted mean of Al/Be even if discordant                  | `load_data()` | Overrides default discordance logic |
# | **C1c**  | Use youngest Al/Be age at stratigraphic level                  | `load_data()` | Prioritizes youngest signal |
# | **C3a**  | Elevation error = max(2 m, 1%)                                 | `load_data()` | Less conservative vertical uncertainty |
# | **C3b**  | Elevation error = max(7.5 m, 3%)                               | `load_data()` | Moderate uncertainty assumption |
# | **C3c**  | Elevation error = max(10 m, 4%)                                | `load_data()` | Most conservative of the three |
# | **C11**  | Use **internal** (not external) age uncertainty (`dtint_LSDn`) | `load_data()` | Assumes different error model |
# | **C4a**  | Fewer spline knots (`+1`)                                 | `run_bayesian_spline()` | Least flexible spline |
# | **C4b**  | More knots (`+5`)                                         | `run_bayesian_spline()` | Moderately flexible spline |
# | **C4c**  | Many knots (`+10`)                                        | `run_bayesian_spline()` | Most flexible spline |
# | **C10a** | Low proposal scale = 10                                   | `run_bayesian_spline()` | Slower MCMC tuning, fine-grained |
# | **C10b** | Proposal scale = 100                                      | `run_bayesian_spline()` | Faster but broader steps |
# | **C10c** | Proposal scale = 200                                      | `run_bayesian_spline()` | Very fast proposals, less stability |
# | **C5a**  | Turns off **Ry** (fit-to-data penalty)                       | `log_likelihood()` | Ignores stratigraphy/spline match |
# | **C5b**  | Ry = 10 (stronger data-fit penalty)                          | `log_likelihood()` | Tighter fit requirement |
# | **C5c**  | Ry = 100 (very strict)                                       | `log_likelihood()` | Aggressive stratigraphic enforcement |
# | **C6a**  | Reduces **Ru** (monotonic penalty) to 1                      | `log_likelihood()` | Allows more curvature |
# | **C6b**  | Ru = 100                                                     | `log_likelihood()` | Enforces stronger monotonic thinning |
# | **C6c**  | Ru = 1000                                                    | `log_likelihood()` | Very strict monotonicity |
# | **C7a**  | Removes **Rh** (highest elevation at oldest)                 | `log_likelihood()` | No peak-age constraint |
# | **C7b**  | Rh = 10                                                      | `log_likelihood()` | Stronger penalty if peak not at oldest age |
# | **C7c**  | Rh = 100                                                     | `log_likelihood()` | Very strict peak-age enforcement |
# | **C8a**  | Removes **Rb** (minimum elevation at youngest)               | `log_likelihood()` | No thinning-to-present constraint |
# | **C8b**  | Rb = 10                                                      | `log_likelihood()` | Stronger thinning constraint |
# | **C8c**  | Rb = 100                                                     | `log_likelihood()` | Enforces thinning strongly at modern age |

In [5]:
# Will carry these tests through the rest of the script
# This has been commented out to allow for the script to find the sensitivity test that maximizes the R-squared value for all of the sites in the analysis, since they are all out of distirbution with one another. 
# SENSITIVITY_TEST = 'X' 

In [6]:
import pandas as pd

# Pull out single assigned sensitivity test for each site, these have been determined separately and are contained in 'bestestsummary.xlsx'
best_df = pd.read_excel('BestSensitivityTest/besttestssummary.xlsx')

# Always pick the first test (split on commas) and remove extra spaces
best_tests_map = (
    best_df
    .set_index('Group')['SensitivityTest']
    .apply(lambda val: val.split(',')[0].strip())
    .to_dict()
)

def load_data(path, SENSITIVITY_TEST):
    data = pd.read_excel(path)

    # Apply basic filtering
    data_in = data.copy()                        # pre-filter dataset
    data = data_in[data_in['Reject'] < 1]        # keep only valid rows
    data_rejected = data_in.loc[data_in.index.difference(data.index)]

    return data_in, data, data_rejected

# Loop through sites with their SINGLE assigned test, print out the assigned sensitivity test below 
for site_id, SENSITIVITY_TEST in best_tests_map.items():
    print(f"▶ Site {site_id} | Test {SENSITIVITY_TEST}")

    # Load data for this specific test
    data_in, data, data_rejected = load_data(
        'icedcurrent_v14_EM.xlsx',
        SENSITIVITY_TEST=SENSITIVITY_TEST
    )

▶ Site 3 | Test C8a
▶ Site 4 | Test C1c
▶ Site 7 | Test C3a
▶ Site 11 | Test C8a
▶ Site 15 | Test C3a
▶ Site 17 | Test C3a
▶ Site 21 | Test C3a
▶ Site 22 | Test C3a
▶ Site 24 | Test C3a
▶ Site 27 | Test C3a
▶ Site 42 | Test C3a
▶ Site 28 | Test C7a
▶ Site 72 | Test C3a
▶ Site 30 | Test C3a
▶ Site 33 | Test C5c
▶ Site 34 | Test C3a
▶ Site 75 | Test C11
▶ Site 37 | Test C3a
▶ Site 39 | Test C4c
▶ Site 40 | Test C3a
▶ Site 45 | Test C3a
▶ Site 46 | Test C3a
▶ Site 47 | Test C3a
▶ Site 78 | Test C3a
▶ Site 79 | Test C3a
▶ Site 80 | Test C3a
▶ Site 48 | Test C6a
▶ Site 49 | Test C4c
▶ Site 50 | Test C6a
▶ Site 52 | Test C3a
▶ Site 53 | Test C3a
▶ Site 54 | Test C4b
▶ Site 55 | Test C3a
▶ Site 56 | Test C3a
▶ Site 57 | Test C4c
▶ Site 59 | Test C3a
▶ Site 60 | Test C6c
▶ Site 61 | Test C11
▶ Site 62 | Test C3a
▶ Site 63 | Test C3a
▶ Site 64 | Test C11
▶ Site 66 | Test C3a
▶ Site 67 | Test C3a
▶ Site 68 | Test C5a
▶ Site 70 | Test C3a
▶ Site 85 | Test C4a
▶ Site 87 | Test C3a
▶ Site 88 | Test

## Bayesian Spline and Bootstrapping Functions 

In [7]:
# Import necessary packages 
import numpy as np
from astropy.stats import bootstrap
from scipy.interpolate import BSpline

# Define number of bootstrap + posterior samples; this will be carried throughout the code
N_BOOTSTRAP_SAMPLES = 1000
N_POSTERIOR_SAMPLES = 1000

n_samples = N_BOOTSTRAP_SAMPLES  # for shaping output arrays and plotting

# Checks if all pairs of age ranges in the DataFrame overlap
def check_pairwise_overlap(df):
    for i in range(len(df)):
        age_i = abs(df.iloc[i]['age'])
        sigma_i = df.iloc[i]['dtint_LSDn']
        lower_i = age_i - 2 * sigma_i
        upper_i = age_i + 2 * sigma_i
        
        for j in range(i + 1, len(df)):
            age_j = abs(df.iloc[j]['age'])
            sigma_j = df.iloc[j]['dtint_LSDn']
            lower_j = age_j - 2 * sigma_j
            upper_j = age_j + 2 * sigma_j
            
            if upper_i < lower_j or upper_j < lower_i:
                return False
    return True

# Calculates weighted mean for a group
def take_weighted_mean(grp):
    wts = (1 / grp.age_err) / (1/grp.age_err).sum()
    grp['age'] = (grp.age * wts).sum()
    grp['age_err'] = grp.age_err.mean()
    grp['elv_comp'] = grp.elv_comp.mean()
    grp['elv_err'] = grp.elv_err.mean()
    return grp.drop_duplicates('age')

# Load and preprocess data
def load_data(path, SENSITIVITY_TEST):
    data = pd.read_excel(path)
    data = data[(data.t_LSDn < 30000) & (data.t_LSDn > 0)].reset_index(drop=True)

    data[['latrnd', 'lonrnd']] = (data[['lat_DD','lon_DD']] * 2).apply(lambda x: pd.Series.round(x,0)) / 2
    grplat = data.groupby('Group')['lat_DD'].mean()
    grplon = data.groupby('Group')['lon_DD'].mean()

    n = 0.05
    data['Group_lat'] = (data['Group'].map(grplat) * n).round(0) / n
    data['Group_lon'] = (data['Group'].map(grplon) * n).round(0) / n

    # Clustering
    coords = data[['lat_DD', 'lon_DD']].to_numpy()
    epsilon = 0.75
    db = DBSCAN(eps=epsilon, min_samples=1, metric='euclidean').fit(coords)
    data['cluster'] = db.labels_

    # Convert to polar stereographic
    target_crs  = 'epsg:3031'
    source_crs = 'epsg:4326'
    latlon_to_polar = pyproj.Transformer.from_crs(source_crs, target_crs)
    x_DD, y_DD = latlon_to_polar.transform(data.lat_DD, data.lon_DD)
    data['x_DD'] = x_DD
    data['y_DD'] = y_DD
    data['t_LSDn'] = data['t_LSDn']
    data['age'] = data.t_LSDn * -1
    data['age_err'] = data.dtext_LSDn

    if SENSITIVITY_TEST == 'C11':
        data['age_err'] = data.dtint_LSDn

    data['age'] = np.round(data.age * 2, -1) / 2

    # Vertical uncertainty
    def get_errs(x):
        if SENSITIVITY_TEST == 'C3a':
            return np.maximum(2, 0.01 * x)
        if SENSITIVITY_TEST == 'C3b':
            return np.maximum(7.5, 0.03 * x)
        if SENSITIVITY_TEST == 'C3c':
            return np.maximum(10, 0.04 * x)
        return np.maximum(10, 0.05 * x)

    elv_err = data.elv_m.map(get_errs)
    data['elv_err'] = np.minimum(data.elv_m_err_1std.fillna(100000), elv_err)
    data['elv_comp'] = data['rel_elv_m'].fillna(data['elv_m'])

    data_in = data.copy()
    data = data_in[data_in.Reject < 1]

    # Remove chlorine and helium, as they are unreliable nuclides 
    data = data[data.cosmo_type != 'Cl']
    data = data[data.cosmo_type != 'He']
    data['cosmo_type'] = data.cosmo_type.replace('e', 'Be')
    data['cosmo_type'] = data.cosmo_type.replace('Bee', 'Be')

    data = data.drop(
        ['thick_cm', 'density', 'what', 'N10_atoms_g', 'delN10_atoms_g',
         'N26_atoms_g', 'delN26_atoms_g', 'Notes-1'], axis=1
    )

    # Split datasets and filter
    datas = []
    for name, grp in data.groupby('Group'):
        splitage = 11700
        data_yng = grp[grp.t_LSDn < splitage]
        data_old = grp[grp.t_LSDn >= splitage]

        if len(data_old) > 0:
            max_yng = data_yng.elv_comp.max()
            data_old = data_old[data_old.elv_comp >= max_yng]

        datas.append(data_yng)
        datas.append(data_old)
        print(str(name), end=' ')

    data = pd.concat(datas, axis=0)

    # Apply Be/Al/C filtering
    savegrps=[]
    for i, (_, grp) in enumerate(data.groupby('Group')):
        for elv, elvgrp in grp.groupby('elv_comp'):
            grp_c = elvgrp[elvgrp.cosmo_type == 'C']
            grp_AlBe = elvgrp[elvgrp.cosmo_type != 'C']
            grp_Be = elvgrp[elvgrp.cosmo_type == 'Be']

            if len(grp_c) > 0:
                oldest_c = grp_c.sort_values('age').iloc[[0]]
                if len(grp_AlBe) > 0:
                    if (abs(oldest_c.age.values) - abs(grp_AlBe.age.values)).max() < 0:
                        print('C', end=', ')
                        savegrp = grp_c
                else:
                    savegrp = grp_c
            else:
                if (len(grp_AlBe) == 1):
                    print('1', end=',')
                    savegrp = grp_AlBe
                else:
                    if check_pairwise_overlap(grp_AlBe):
                        print('concordant', end=',')
                        savegrp = take_weighted_mean(grp_AlBe)
                    else:
                        print('Be', end=',')
                        savegrp = take_weighted_mean(grp_Be)

                if SENSITIVITY_TEST == 'C1b':
                    savegrp = take_weighted_mean(grp_AlBe)
                if SENSITIVITY_TEST == 'C1c':
                    savegrp = grp_AlBe.sort_values('age').iloc[[-1]]

            savegrps.append(savegrp)

    data = pd.concat(savegrps)

    # Clamp below-zero elevations
    data['elv_comp'] = data['elv_comp'].mask(data['elv_comp'] <= 0, 0.1)
    data['DOI_oldest'] = data.DOI.str.split(',').str[0]

    cluster_weights = 1 / data.groupby(['cluster'])['Group'].nunique()
    cluster_weights = cluster_weights / cluster_weights.sum()
    data['cluster_weights'] = data['cluster'].map(cluster_weights)

    idx = data_in.index[~data_in.index.isin(data.index)].dropna()
    data_rejected = data_in.loc[idx]
    return data_in, data, data_rejected


# Bayesian spline
def run_bayesian_spline(ages, age_uncertainties, elevations, elevation_uncertainties, 
                        n_samples=1000, SENSITIVITY_TEST=None, xstep=5):
    degree = 3
    num = len(ages) + degree
    if SENSITIVITY_TEST == 'C4a':
        num = len(ages) + 1
    if SENSITIVITY_TEST == 'C4b':
        num = len(ages) + 5
    if SENSITIVITY_TEST == 'C4c':
        num = len(ages) + 10

    knot_locs = np.linspace(ages.min(), ages.max(), num=num)

    def create_bspline_basis(x, knots, degree):
        n_knots = len(knots)
        extended_knots = np.concatenate(([knots[0]] * degree, knots, [knots[-1]] * degree))
        basis = np.zeros((len(x), n_knots + degree - 1))
        for i in range(n_knots + degree - 1):
            coeff = np.zeros(n_knots + degree - 1)
            coeff[i] = 1
            spline = BSpline(extended_knots, coeff, degree)
            basis[:, i] = spline(x)
        return basis

    B = create_bspline_basis(ages, knot_locs, degree)

    data = np.vstack([ages, age_uncertainties]).T
    bootstrapped_data = bootstrap(data, bootnum=n_samples)

    bspline_basis_bootstrapped = []
    for i in range(n_samples):
        ages_i = bootstrapped_data[i, :, 0]
        basis_i = create_bspline_basis(ages_i, knot_locs, degree)
        bspline_basis_bootstrapped.append(basis_i)

    bspline_basis_bootstrapped = np.stack(bspline_basis_bootstrapped)

    coeffs = np.zeros(B.shape[1])
    burn_in = 1000
    coeff_samples = np.zeros((n_samples, B.shape[1]))

    def log_likelihood(coeffs, SENSITIVITY_TEST):
        mu = B @ coeffs
        model_ages = np.interp(elevations, mu, ages)
        age_residuals = (ages - model_ages) ** 2 / age_uncertainties ** 2
        elevation_residuals = (elevations - mu) ** 2 / elevation_uncertainties ** 2
        wrss = np.sum(age_residuals + elevation_residuals)
        ll = -0.5 * wrss

        Ry, Ru, Rh, Rb = 1, 10, 1, 1
        if 'a' in SENSITIVITY_TEST:
            if '5' in SENSITIVITY_TEST: Ry = 0
            if '6' in SENSITIVITY_TEST: Ru = 1
            if '7' in SENSITIVITY_TEST: Rh = 0
            if '8' in SENSITIVITY_TEST: Rb = 0 
        if 'b' in SENSITIVITY_TEST:
            if '5' in SENSITIVITY_TEST: Ry = 10
            if '6' in SENSITIVITY_TEST: Ru = 100
            if '7' in SENSITIVITY_TEST: Rh = 10
            if '8' in SENSITIVITY_TEST: Rb = 10 
        if 'c' in SENSITIVITY_TEST:
            if '5' in SENSITIVITY_TEST: Ry = 100
            if '6' in SENSITIVITY_TEST: Ru = 1000
            if '7' in SENSITIVITY_TEST: Rh = 100
            if '8' in SENSITIVITY_TEST: Rb = 100 

        age_factor = (ages - ages.min()) / (ages.max() - ages.min())
        elevation_diff = elevations - mu
        age_scaled_diff = age_factor / age_uncertainties
        elevation_scaled_diff = elevation_diff / elevation_uncertainties
        soft_penalty = -np.sum(np.maximum(0, elevation_scaled_diff + age_scaled_diff) ** 2) * Ry
        non_decreasing_penalty = -np.sum(np.maximum(0, coeffs[:-1] - coeffs[1:]) ** 2) * Ru
        highest_sample_age = ages[np.argmax(elevations)]
        highest_elevation_penalty = -((mu[np.argmax(ages == highest_sample_age)] - elevations.max()) ** 2)  * Rh
        lowest_elevation_penalty = -((mu[0] - elevations.min()) ** 2)  * Rb

        return ll + soft_penalty + non_decreasing_penalty + highest_elevation_penalty + lowest_elevation_penalty

    proposal_scale = 60
    if SENSITIVITY_TEST == 'C10a': proposal_scale = 10
    if SENSITIVITY_TEST == 'C10b': proposal_scale = 100
    if SENSITIVITY_TEST == 'C10c': proposal_scale = 200

    for i in range(n_samples + burn_in):
        for j in range(len(coeffs)):
            current_coeff = coeffs[j]
            proposed_coeff = current_coeff + np.random.normal(0, proposal_scale)
            coeffs[j] = proposed_coeff
            proposed_log_likelihood = log_likelihood(coeffs, SENSITIVITY_TEST)
            coeffs[j] = current_coeff
            current_log_likelihood = log_likelihood(coeffs, SENSITIVITY_TEST)

            log_acceptance_ratio = proposed_log_likelihood - current_log_likelihood
            acceptance_ratio = np.exp(log_acceptance_ratio)
            if np.random.rand() < acceptance_ratio:
                coeffs[j] = proposed_coeff

        if i >= burn_in:
            coeff_samples[i - burn_in] = coeffs

    coeff_mean = np.mean(coeff_samples, axis=0)
    ages_high_res = np.arange(ages.min()-xstep, ages.max()+xstep, xstep)
    B_high_res = create_bspline_basis(ages_high_res, knot_locs, degree)
    spline_samples_high_res = B_high_res @ coeff_samples.T
    spline_mean_high_res = np.mean(spline_samples_high_res, axis=1)
    spline_lower_high_res = np.percentile(spline_samples_high_res, 2.5, axis=1)
    spline_upper_high_res = np.percentile(spline_samples_high_res, 97.5, axis=1)

    def create_bspline_basis_derivative(x, knots, degree):
        n_knots = len(knots)
        extended_knots = np.concatenate(([knots[0]] * degree, knots, [knots[-1]] * degree))
        basis_derivative = np.zeros((len(x), n_knots + degree - 1))
        for i in range(n_knots + degree - 1):
            coeff = np.zeros(n_knots + degree - 1)
            coeff[i] = 1
            spline = BSpline(extended_knots, coeff, degree)
            basis_derivative[:, i] = spline.derivative()(x)
        return basis_derivative

    B_derivative_high_res = create_bspline_basis_derivative(ages_high_res, knot_locs, degree)
    rate_of_change_samples_high_res = B_derivative_high_res @ coeff_samples.T
    rate_mean_high_res = np.mean(rate_of_change_samples_high_res, axis=1)
    rate_lower_high_res = np.percentile(rate_of_change_samples_high_res, 2.5, axis=1)
    rate_upper_high_res = np.percentile(rate_of_change_samples_high_res, 97.5, axis=1)

    return (ages_high_res, spline_mean_high_res, spline_lower_high_res, spline_upper_high_res,
            spline_samples_high_res, rate_mean_high_res, rate_upper_high_res, rate_lower_high_res,
            rate_of_change_samples_high_res)


## Regression and Plotting  

In [8]:
# This code block will spit out of a lot of processing text as it runs 

# Import necessary packages 
import re
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr
from sklearn.metrics import r2_score

# Defining function to plot elevation-time regression
def plot_tem_regression(x, y, xerrs, yerrs,
                        ages_post, elv_post, elv_lo_bound, elv_hi_bound,
                        elv_rate_post, elv_rate_lo_bound, elv_rate_hi_bound,
                        title, color, fig=None, axes=None, placename=None,
                        yhi=None, xhi=None, LGM_lim=None,
                        elv_post_samples=None, elv_post_rate_samples=None):
    # Clip elevation arrays to prevent plotting values below 0
    elv_post = np.clip(elv_post, 0, None)
    elv_lo_bound = np.clip(elv_lo_bound, 0, None)
    elv_hi_bound = np.clip(elv_hi_bound, 0, None)

    if elv_post_samples is not None:
        elv_post_samples = np.clip(elv_post_samples, 0, None)

    # Truncate all arrays to match the same minimum length
    min_len = min(len(ages_post), len(elv_post), len(elv_lo_bound), len(elv_hi_bound))
    ages_post = ages_post[:min_len]
    elv_post = elv_post[:min_len]
    elv_lo_bound = elv_lo_bound[:min_len]
    elv_hi_bound = elv_hi_bound[:min_len]
    elv_rate_post = elv_rate_post[:min_len]
    elv_rate_lo_bound = elv_rate_lo_bound[:min_len]
    elv_rate_hi_bound = elv_rate_hi_bound[:min_len]

    if elv_post_samples is not None:
        elv_post_samples = elv_post_samples[:min_len, :]
    if elv_post_rate_samples is not None:
        elv_post_rate_samples = elv_post_rate_samples[:min_len, :]

    if fig is None or axes is None:
        fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(18, 6))

    # Left subplot: Elevation over time
    ax0 = axes[0]
    ax0.grid(False)

    # Plot data points with uncertainty
    ax0.errorbar(x, y, xerr=xerrs, yerr=yerrs, fmt='o',
                 color='black', markersize=5, zorder=3)

    # Interpolate model predictions at observed ages (no extrapolation)
    pred = np.interp(x, ages_post, elv_post, left=np.nan, right=np.nan)

    # Mask out NaNs
    mask = ~np.isnan(pred) & ~np.isnan(y)
    if np.sum(mask) >= 2:  # Need at least 2 points
        x_m = x[mask]
        y_m = y[mask]
        pred_m = pred[mask]
        err_m = yerrs[mask]

        # Pearson r
        r = np.corrcoef(x_m, y_m)[0, 1]

        # R²
        r2 = r2_score(y_m, pred_m)

        # Weighted R² and WRSS
        w = 1.0 / np.clip(err_m, 1e-6, None)**2
        wrss = np.sum(w * (y_m - pred_m)**2)
        ybar_w = np.average(y_m, weights=w)
        ss_tot_w = np.sum(w * (y_m - ybar_w)**2)
        wr2 = 1.0 - wrss / ss_tot_w if ss_tot_w > 0 else np.nan
    else:
        r, r2, wr2, wrss = np.nan, np.nan, np.nan, np.nan

    # Annotate metrics
    ax0.text(
        0.05, 0.95,
        f'$r$ = {r:.2f},  $R^2$ = {r2:.2f},  $W\\!R^2$ = {wr2:.2f},  WRSS = {wrss:.1f}',
        transform=ax0.transAxes,
        ha='left', va='top',
        fontsize=12,
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.7),
    )

    # Plot bootstrap samples (left panel)
    if elv_post_samples is not None and elv_post_samples.ndim == 2:
        for i in range(0, elv_post_samples.shape[1], 10):
            ax0.plot(ages_post, elv_post_samples[:, i], color=color, alpha=0.05,
                     linewidth=0.5, zorder=1)

    # Expand CI by measurement error
    meas_margin = np.nanmean(yerrs) if yerrs is not None and len(yerrs) > 0 else 0
    elv_lo_bound_plot = np.clip(elv_lo_bound - meas_margin, 0, None)
    elv_hi_bound_plot = np.clip(elv_hi_bound + meas_margin, 0, None)

    min_len_ci = min(len(ages_post), len(elv_lo_bound_plot), len(elv_hi_bound_plot))
    ages_post = ages_post[:min_len_ci]
    elv_post = elv_post[:min_len_ci]
    elv_lo_bound_plot = elv_lo_bound_plot[:min_len_ci]
    elv_hi_bound_plot = elv_hi_bound_plot[:min_len_ci]

    ax0.fill_between(ages_post, elv_lo_bound_plot, elv_hi_bound_plot,
                     color=color, alpha=0.3, zorder=2)
    ax0.plot(ages_post, elv_post, color=color, lw=3, label=placename, zorder=4)

    # Bootstrap range envelope
    if elv_post_samples is not None:
        min_bootstrap = np.nanmin(elv_post_samples, axis=1)
        max_bootstrap = np.nanmax(elv_post_samples, axis=1)
        ax0.fill_between(ages_post, min_bootstrap, max_bootstrap,
                         color=color, alpha=0.15, label='Bootstrap range')

    # LGM limit line
    if LGM_lim is not None:
        ax0.axhline(LGM_lim, color=color, linestyle='dashed',
                    linewidth=1, label='LGM limit')

    ax0.set_xlabel('Age (BP)', fontsize=14)
    ax0.set_ylabel('Elevation above present-day ice (m)', fontsize=14)
    ax0.set_ylim(0, yhi)
    ax0.set_xlim(xhi, 0)
    ax0.set_title(title, fontsize=16)
    ax0.legend()

    # Right subplot: Rate of change
    ax1 = axes[1]
    ax1.grid(False)

    if elv_post_rate_samples is not None and elv_post_rate_samples.ndim == 2:
        for i in range(0, elv_post_rate_samples.shape[1], 10):
            ax1.plot(ages_post, elv_post_rate_samples[:, i],
                     color=color, alpha=0.05, linewidth=0.5, zorder=1)

    ax1.plot(ages_post, elv_rate_post, color=color, lw=3, zorder=3)
    ax1.fill_between(ages_post, elv_rate_lo_bound, elv_rate_hi_bound,
                     color=color, alpha=0.3, zorder=2)

    if LGM_lim:
        ax1.axhline(0, color='gray', linewidth=1, linestyle='dashed', zorder=0)

    ax1.set_xlabel('Age (ka)', fontsize=14)
    ax1.set_ylabel('Rate of ice thinning (m/kyr)', fontsize=14)
    ax1.set_ylim(0, None)
    ax1.set_xlim(xhi, 0)

    return fig, axes, r, r2, wr2, wrss

all_dses = []

# 1) Loop across all sites and sensitivity tests 
all_metrics = []
ncolors = ['blue','green','purple','gold','orange','olive',
           'darkorchid','teal','pink','dodgerblue','turquoise',
           'saddlebrown','seagreen','mediumpurple','honeydew']

for site_id, SENSITIVITY_TEST in best_tests_map.items():
    print(f"=== Running site {site_id} with test {SENSITIVITY_TEST} ===")

    # Reset per‐site list
    dses = []

    # Load & filter data for this site/test
    data_in, data, data_rejected = load_data(
        'icedcurrent_v14_EM.xlsx',
        SENSITIVITY_TEST=SENSITIVITY_TEST
    )
    data      = data[data['Group'] == site_id]
    data_rej  = data_rejected[data_rejected['Group'] == site_id]
    if data.empty:
        print(f"No data for site {site_id}")
        continue

    # Loop over clusters & site‐groups
    for clustername, cluster in data.sort_values('Group').groupby('cluster'):
        for i, (name, grp) in enumerate(cluster.groupby('Group')):
            if len(grp) < 4:
                continue

            # If only one point, add zero‐age
            if len(grp) == 1:
                zeroage = grp['Best guess at when thickness reaches present'].max() * -1 - 20
                newrow = grp.iloc[-1].copy()
                newrow['elv_comp'] = 0
                newrow['elv_err']  = 0.1
                newrow['age']      = zeroage
                newrow['age_err']  = 0.1
                grp = pd.concat([grp, pd.DataFrame(newrow).T], ignore_index=True)

            # Prepare x,y arrays
            x, y, xerrs, yerrs = make_data_xy(grp)

            # Run spline
            outputs = run_bayesian_spline(
                x, xerrs, y, yerrs,
                n_samples=1000,
                SENSITIVITY_TEST=SENSITIVITY_TEST
            )
            x_fine, y_fine, lower_bound, upper_bound, bootstrap_samples = outputs[:5]
            mean_rate, rate_upper, rate_lower, rate_of_change_samples = outputs[5:]

            # Convert to m/kyr
            mean_rate *= 200
            rate_upper *= 200
            rate_lower *= 200
            rate_of_change_samples *= 200

            # Build full time arrays
            fullmin  = np.minimum(0, x_fine.min())
            fulltime = np.arange(fullmin, 20000.000001, 5)
            lo = fulltime[fulltime < x_fine.min()]
            hi = fulltime[fulltime > x_fine.max()]
            if len(lo) and round(lo[-1],4) == x_fine[0]:
                lo = lo[:-1]
            if len(hi) and round(hi[0],4) == x_fine[-1]:
                hi = hi[1:]

            loz      = np.clip(np.linspace(0, y_fine.min(), len(lo)), 0, None)
            loz_rate = loz.copy()
            hiz      = np.zeros(len(hi)) * np.nan
            hiz_rate = hiz.copy()

            # Piecewise lower if needed
            hitszero = grp['Best guess at when thickness reaches present'].max()
            if y_fine.min() > 0:
                loz = np.linspace(0, y_fine.min(), len(lo))
                if hitszero > 0:
                    changelen = len(lo[lo > hitszero])
                    loz = np.concatenate([
                        np.zeros(len(lo[lo <= hitszero])),
                        np.linspace(0, y_fine.min(), changelen)
                    ])
                loz_rate = np.gradient(loz) * 200

            # Upper if LGM applies
            if 'YES' in grp['upper bound LGM ice extent?'].mode()[0]:
                LGM_mean = grp['rel_elv_LGM_m'].mean()
                if LGM_mean > 0:
                    hiz = np.linspace(y_fine.max(), LGM_mean, max(len(hi),2))
                    hiz_rate = np.gradient(hiz) * 200

            # Concatenate
            ages_post         = np.concatenate([lo, x_fine, hi])
            elv_post          = np.concatenate([loz, y_fine, hiz])
            elv_lo_bound      = np.concatenate([loz, lower_bound, hiz])
            elv_hi_bound      = np.concatenate([loz, upper_bound, hiz])
            elv_rate_post     = np.concatenate([loz_rate, mean_rate, hiz_rate])
            elv_rate_lo_bound = np.concatenate([loz_rate, rate_lower, hiz_rate])
            elv_rate_hi_bound = np.concatenate([loz_rate, rate_upper, hiz_rate])

            # Bootstrap samples
            nsamp = bootstrap_samples.shape[1]
            loz_samp      = loz[:,None] * np.ones((1,nsamp))
            loz_rate_samp = loz_rate[:,None] * np.ones((1,nsamp))
            hiz_samp      = hiz[:,None] * np.ones((1,nsamp))
            hiz_rate_samp = hiz_rate[:,None] * np.ones((1,nsamp))

            elv_post_samples      = np.concatenate([loz_samp, bootstrap_samples, hiz_samp])
            elv_post_rate_samples = np.concatenate([loz_rate_samp, rate_of_change_samples, hiz_rate_samp])

            # Trim to common length
            mlen = min(len(ages_post), elv_post.shape[0], elv_post_samples.shape[0])
            ages_post              = ages_post[:mlen]
            elv_post               = elv_post[:mlen]
            elv_lo_bound           = elv_lo_bound[:mlen]
            elv_hi_bound           = elv_hi_bound[:mlen]
            elv_rate_post          = elv_rate_post[:mlen]
            elv_rate_lo_bound      = elv_rate_lo_bound[:mlen]
            elv_rate_hi_bound      = elv_rate_hi_bound[:mlen]
            elv_post_samples       = elv_post_samples[:mlen]
            elv_post_rate_samples  = elv_post_rate_samples[:mlen]

            # Build per-site xarray Dataset
            ds = xr.Dataset(
                data_vars=dict(
                    elv_post=(['age','site'], elv_post[:,None]),
                    elv_rate_post=(['age','site'], elv_rate_post[:,None]),
                    elv_post_samples=(['age','samples','site'], elv_post_samples[:,:,None]),
                    elv_post_rate_samples=(['age','samples','site'], elv_post_rate_samples[:,:,None]),
                ),
                coords=dict(
                    age=ages_post,
                    site=[name],
                    lat=(['site'], [grp.lat_DD.mean()]),
                    lon=(['site'], [grp.lon_DD.mean()]),
                    samples=np.arange(nsamp),
                    cluster=grp.cluster.mean(),
                    cluster_weight=grp.cluster_weights.mean(),
                ),
            ).interp(age=np.arange(0,20000+1,2))

            # Append to lists
            dses.append(ds)
            all_dses.append(ds)

            # Plot & compute metrics
            fig, axes, r_val, r2_val, wr2_val, wrss_val = plot_tem_regression(
                x, y, xerrs, yerrs,
                ages_post, elv_post, elv_lo_bound, elv_hi_bound,
                elv_rate_post, elv_rate_lo_bound, elv_rate_hi_bound,
                title=name,
                color=ncolors[i],
                placename=name,
                yhi=None, xhi=None, LGM_lim=None,
                elv_post_samples=elv_post_samples,
                elv_post_rate_samples=elv_post_rate_samples
            )
            all_metrics.append((clustername, name, r_val, r2_val, wr2_val, wrss_val))

            # Save figure (cast name to string for regex)
            safe = re.sub(r'[\\/*?:"<>|]', "", str(name)).replace(" ", "_")
            outdir = 'spline'
            os.makedirs(outdir, exist_ok=True)
            fig.savefig(os.path.join(outdir, f"{safe}_{SENSITIVITY_TEST}.png"),
                        dpi=300, bbox_inches='tight')
            plt.close(fig)


=== Running site 3 with test C8a ===
1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25 26 27 28 29 30 31 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47 48 49 50 51 52 53 54 55 56 57 58 59 60 61 62 63 64 65 66 67 68 69 70 71 72 75 76 78 79 80 81 82 83 84 85 86 87 88 89 90 91 92 93 94 95 96 97 98 99 100 101 102 103 104 105 106 107 108 109 110 111 112 113 114 116 117 118 119 120 121 122 123 124 125 126 127 128 129 130 131 132 133 134 135 136 137 142 143 144 145 146 147 148 149 150 151 152 153 154 155 156 157 158 159 160 161 162 163 164 165 166 167 168 169 concordant,concordant,concordant,1,1,1,1,1,concordant,concordant,concordant,concordant,concordant,Be,concordant,concordant,concordant,1,1,concordant,concordant,1,concordant,concordant,concordant,1,1,Be,concordant,concordant,Be,concordant,1,1,concordant,concordant,concordant,concordant,concordant,1,1,1,concordant,concordant,1,concordant,1,concordant,1,1,concordant,1,concordant,1,1,1,concordant,concordant,1,concordant,concord

In [9]:
# Define sites to exclude based on R-squared values (anything below 0.5)
exclude_sites = {3, 39, 48, 53, 55, 63, 66, 87, 88, 95, 103, 105, 123, 124, 135, 163, 165, 168, 169}

ds_sites = xr.concat(all_dses, dim='site')
print("Sites in ds_sites before filtering:", ds_sites['site'].values)

Sites in ds_sites before filtering: [  3   4   7  11  15  17  21  22  24  27  42  28  72  30  33  34  75  37
  39  40  45  46  47  78  79  80  48  49  50  52  53  54  55  56  57  59
  60  61  62  63  64  66  67  68  70  85  87  88  90  91  95  97 111 103
 114 105 116 144 121 122 123 142 124 125 128 129 135 145 148 149 151 152
 156 163 164 165 166 168 169 161]


In [10]:
# Extract mean thickness over time from all the sites with an R-squared >0.5 
records = []
for sid in ds_sites['site'].values:
    if sid in exclude_sites:
        continue

    elv   = ds_sites['elv_post'].sel(site=sid)
    ages  = elv['age'].values
    vals  = elv.values
    latm  = float(ds_sites['lat'].sel(site=sid))
    lonm  = float(ds_sites['lon'].sel(site=sid))

    records.append(pd.DataFrame({
        'Site':      sid,
        'Age':       ages,
        'Thickness': vals,
        'Latm':      latm,
        'Lonm':      lonm
    }))

th_df = pd.concat(records, ignore_index=True)

# Pad endpoints so each site has Age = 20000 and Age = 0
def pad_endpoints(df):
    out = []
    for sid, grp in df.groupby('Site'):
        grp = grp.copy()
        min_age, max_age = grp['Age'].min(), grp['Age'].max()
        lat0, lon0       = grp[['Latm','Lonm']].iloc[0]

        if 20000 not in grp['Age'].values:
            t0 = grp.loc[grp['Age']==min_age, 'Thickness'].iloc[0]
            grp = grp.append({
                'Site': sid, 'Age': 20000, 'Thickness': t0,
                'Latm': lat0, 'Lonm': lon0
            }, ignore_index=True)

        if 0 not in grp['Age'].values:
            t1 = grp.loc[grp['Age']==max_age, 'Thickness'].iloc[0]
            grp = grp.append({
                'Site': sid, 'Age': 0, 'Thickness': t1,
                'Latm': lat0, 'Lonm': lon0
            }, ignore_index=True)

        out.append(grp)
    full = pd.concat(out, ignore_index=True)
    return full.sort_values(['Site','Age'], ascending=[True, False])

th_df = pad_endpoints(th_df)

# Save output in a CSV 
output_path = "thicknesstimeseriesmean.csv"
th_df.to_csv(output_path, index=False)
print(f"Wrote: {output_path}")

Wrote: thicknesstimeseriesmean.csv


## Sampling Upper and Lower Uncertainty Bounds

In [11]:
# Initialize empty DataFrames
df_lower = pd.DataFrame()
df_upper = pd.DataFrame()

# Grab shared age axis (1D)
age_values = ds_sites['elv_post'].coords['age'].values

# Loop through sites, skipping excluded ones
for site in ds_sites.site.values:
    if site in exclude_sites:
        continue

    # Extract bootstrap samples for this site: shape (n_samples, n_ages)
    samples = ds_sites['elv_post_samples'].sel(site=site).values

    # If samples are transposed, fix it
    if samples.shape[1] != len(age_values):
        samples = samples.T

    # Sanity check
    assert samples.shape[1] == len(age_values), (
        f"Mismatch for site {site}: samples shape = {samples.shape}, "
        f"age_values length = {len(age_values)}"
    )

    # Compute 2.5th and 97.5th percentiles
    lower_vals = np.nanpercentile(samples, 2.5, axis=0)
    upper_vals = np.nanpercentile(samples, 97.5, axis=0)

    # Get site coordinates
    lat = float(ds_sites['lat'].sel(site=site).values)
    lon = float(ds_sites['lon'].sel(site=site).values)

    # Build DataFrame rows
    df_site_lower = pd.DataFrame({
        'Site':      site,
        'Age':       age_values,
        'Thickness': lower_vals,
        'Latm':      lat,
        'Lonm':      lon
    })
    df_lower = pd.concat([df_lower, df_site_lower], ignore_index=True)

    df_site_upper = pd.DataFrame({
        'Site':      site,
        'Age':       age_values,
        'Thickness': upper_vals,
        'Latm':      lat,
        'Lonm':      lon
    })
    df_upper = pd.concat([df_upper, df_site_upper], ignore_index=True)

# Pad endpoints so each site has Age = 20000 and Age = 0
def pad(df):
    padded = []
    for site in df['Site'].unique():
        s_df = df[df['Site'] == site].copy()
        ages = s_df['Age'].values

        if 20000 not in ages:
            first = s_df.loc[s_df['Age'].idxmin()]
            padded.append({
                'Site':      site,
                'Age':       20000,
                'Thickness': first['Thickness'],
                'Latm':      first['Latm'],
                'Lonm':      first['Lonm']
            })
        if 0 not in ages:
            last = s_df.loc[s_df['Age'].idxmax()]
            padded.append({
                'Site':      site,
                'Age':       0,
                'Thickness': last['Thickness'],
                'Latm':      last['Latm'],
                'Lonm':      last['Lonm']
            })
    if padded:
        df = pd.concat([df, pd.DataFrame(padded)], ignore_index=True)
    return df.sort_values(['Site','Age'], ascending=[True, False]).reset_index(drop=True)

df_lower = pad(df_lower)
df_upper = pad(df_upper)

# Save out to CSV
df_lower.to_csv("thicknesstimeserieslowerCI.csv", index=False)
df_upper.to_csv("thicknesstimeseriesupperCI.csv", index=False)

# Extra Code Bits

In [8]:
# # Define number of samples globally
# N_BOOTSTRAP_SAMPLES = 1000
# N_POSTERIOR_SAMPLES = 1000
# n_samples = N_BOOTSTRAP_SAMPLES  # for shaping output arrays and plotting

# def check_pairwise_overlap(df):
#     # Iterate through each row in the DataFrame 
#     for i in range(len(df)):
#         age_i = abs(df.iloc[i]['age'])
#         sigma_i = df.iloc[i]['dtint_LSDn']
#         lower_i = age_i - 2 * sigma_i
#         upper_i = age_i + 2 * sigma_i
        
#         # Compare it with every other row
#         for j in range(i + 1, len(df)):
#             age_j = abs(df.iloc[j]['age'])
#             sigma_j = df.iloc[j]['dtint_LSDn']
#             lower_j = age_j - 2 * sigma_j
#             upper_j = age_j + 2 * sigma_j
            
#             # Check if there's no overlap between the two ranges
#             if upper_i < lower_j or upper_j < lower_i:
#                 return False  # No overlap between at least one pair

#     return True  # All data pairs overlap

# def take_weighted_mean(grp):
#     wts = (1 / grp.age_err) / (1/grp.age_err).sum()
#     grp['age']      = (grp.age * wts).sum()
#     grp['age_err']  = grp.age_err.mean()
#     grp['elv_comp'] = grp.elv_comp.mean()
#     grp['elv_err']  = grp.elv_err.mean()
#     return grp.drop_duplicates('age')

# def load_data(path, SENSITIVITY_TEST=SENSITIVITY_TEST):
#     data = pd.read_excel(path)

#     # Filter out ages outside of the 0-30,000 year range 
#     data = data[(data.t_LSDn < 30000) & (data.t_LSDn > 0)].reset_index(drop=True)
    
#     # Round lat/lon to nearest degree to group data
#     data[['latrnd', 'lonrnd']] = (data[['lat_DD','lon_DD']] * 2).apply(lambda x: pd.Series.round(x,0)) / 2

#     # Get mean of each group's lat and lon
#     grplat = data.groupby('Group')['lat_DD'].mean()
#     grplon = data.groupby('Group')['lon_DD'].mean()
#     n = 0.05
#     data['Group_lat'] = (data['Group'].map(grplat) * n).round(0) / n
#     data['Group_lon'] = (data['Group'].map(grplon) * n).round(0) / n

#     # DBSCAN
#     coords = data[['lat_DD', 'lon_DD']].to_numpy()
#     data['cluster'] = DBSCAN(eps=0.75, min_samples=1).fit_predict(coords)

#     # Polar stereographic
#     transformer = pyproj.Transformer.from_crs('epsg:4326','epsg:3031', always_xy=True)
#     data['x_DD'], data['y_DD'] = transformer.transform(data.lon_DD.values, data.lat_DD.values)

#     # Ages & errors
#     data['age']     = -data.t_LSDn
#     data['age_err'] = data.dtext_LSDn
#     if SENSITIVITY_TEST == 'C11':
#         data['age_err'] = data.dtint_LSDn
#     data['age'] = np.round(data.age * 2, -1) / 2

#     def get_errs(x):
#         if SENSITIVITY_TEST == 'C3a': return np.maximum(2,   0.01 * x)
#         if SENSITIVITY_TEST == 'C3b': return np.maximum(7.5, 0.03 * x)
#         if SENSITIVITY_TEST == 'C3c': return np.maximum(10,  0.04 * x)
#         return np.maximum(10, 0.05 * x)

#     elv_err = data.elv_m.map(get_errs)
#     data['elv_err']  = np.minimum(data.elv_m_err_1std.fillna(1e6), elv_err)
#     data['elv_comp'] = data.rel_elv_m.fillna(data.elv_m)

#     # Drop rejects & unwanted isotopes
#     data_in = data.copy()
#     data = data[(data.Reject < 1) & (~data.cosmo_type.isin(['Cl','He']))]
#     data['cosmo_type'] = data.cosmo_type.replace({'e':'Be','Bee':'Be'})
#     drop_cols = ['thick_cm','density','what','N10_atoms_g','delN10_atoms_g',
#                  'N26_atoms_g','delN26_atoms_g','Notes-1']
#     data = data.drop(columns=[c for c in drop_cols if c in data])

#     # Pre-/post-11.7 ka split
#     pieces = []
#     for name, grp in data.groupby('Group'):
#         yng = grp[grp.t_LSDn < 11700]
#         old = grp[grp.t_LSDn >= 11700]
#         if not old.empty:
#             max_y = yng.elv_comp.max()
#             old   = old[old.elv_comp >= max_y]
#         pieces += [yng, old]
#     data = pd.concat(pieces, ignore_index=True)

#     # Concordance & weighted means
#     savegrps = []
#     for name, grp in data.groupby('Group'):
#         for elv, elvgrp in grp.groupby('elv_comp'):
#             grp_c    = elvgrp[elvgrp.cosmo_type == 'C']
#             grp_AlBe = elvgrp[elvgrp.cosmo_type != 'C']
#             grp_Be   = elvgrp[elvgrp.cosmo_type == 'Be']
#             savegrp  = None

#             if len(grp_c) > 0:
#                 oldest_c_age = abs(grp_c.sort_values('age').iloc[0]['age'])
#                 if len(grp_AlBe) > 0:
#                     youngest_AlBe_age = abs(grp_AlBe['age']).min()
#                     if oldest_c_age < youngest_AlBe_age:
#                         print('C', end=', ')
#                         savegrp = grp_c
#                 else:
#                     savegrp = grp_c
#             else:
#                 if len(grp_AlBe) == 1:
#                     print('1', end=', ')
#                     savegrp = grp_AlBe
#                 else:
#                     if check_pairwise_overlap(grp_AlBe):
#                         print('concordant', end=', ')
#                         savegrp = take_weighted_mean(grp_AlBe)
#                     else:
#                         print('Be', end=', ')
#                         savegrp = take_weighted_mean(grp_Be)
#                 if SENSITIVITY_TEST == 'C1b':
#                     savegrp = take_weighted_mean(grp_AlBe)
#                 if SENSITIVITY_TEST == 'C1c':
#                     savegrp = grp_AlBe.sort_values('age').iloc[[-1]]
#             if savegrp is not None:
#                 savegrps.append(savegrp)

#     data = pd.concat(savegrps, ignore_index=True)
#     data['elv_comp']    = data.elv_comp.mask(data.elv_comp <= 0, 0.1)
#     data['DOI_oldest']  = data.DOI.str.split(',').str[0]
#     cw = 1 / data.groupby('cluster')['Group'].nunique()
#     cw = cw / cw.sum()
#     data['cluster_weights'] = data.cluster.map(cw)
#     rejected_idx = data_in.index.difference(data.index)
#     data_rejected = data_in.loc[rejected_idx]

#     return data_in, data, data_rejected
# def run_bayesian_spline(
#     ages, age_uncertainties, elevations, elevation_uncertainties,
#     n_samples, SENSITIVITY_TEST, xstep=5
# ):
#     import numpy as np
#     from scipy.interpolate import BSpline

#     # Spline order & knot count
#     degree = 3
#     num = len(ages) + degree
#     if SENSITIVITY_TEST == 'C4a': num = len(ages) + 1
#     if SENSITIVITY_TEST == 'C4b': num = len(ages) + 5
#     if SENSITIVITY_TEST == 'C4c': num = len(ages) + 10
#     knot_locs = np.linspace(ages.min(), ages.max(), num=num)

#     # Build B-spline basis
#     def create_bspline_basis(x, knots, degree):
#         extended = np.concatenate(([knots[0]]*degree, knots, [knots[-1]]*degree))
#         n_bases = len(knots) + degree - 1
#         B = np.zeros((len(x), n_bases))
#         for i in range(n_bases):
#             coeff = np.zeros(n_bases); coeff[i] = 1
#             spline = BSpline(extended, coeff, degree)
#             B[:, i] = spline(x)
#         return B

#     B = create_bspline_basis(ages, knot_locs, degree)

#     # Bootstrap block (unchanged)
#     from astropy.stats import bootstrap
#     data_stack = np.vstack([ages, age_uncertainties]).T
#     bootstrapped_data = bootstrap(data_stack, bootnum=n_samples)

#     # Metropolis-Hastings setup
#     coeffs = np.zeros(B.shape[1])
#     burn_in = 1000
#     coeff_samples = np.zeros((n_samples, B.shape[1]))

#     # Full log-likelihood with soft penalties
#     def log_likelihood(coeffs):
#         mu = B @ coeffs

#         # Interpolate model ages
#         model_ages = np.interp(elevations, mu, ages)

#         # WRSS in age/elev space
#         age_resid = (ages - model_ages)**2 / age_uncertainties**2
#         elev_resid = (elevations - mu)**2 / elevation_uncertainties**2
#         wrss = np.sum(age_resid + elev_resid)
#         ll = -0.5 * wrss

#         # Sensitivity-based penalty scales
#         Ry, Ru, Rh, Rb = 1, 10, 1, 1
#         if 'a' in SENSITIVITY_TEST:
#             if '5' in SENSITIVITY_TEST: Ry = 0
#             if '6' in SENSITIVITY_TEST: Ru = 1
#             if '7' in SENSITIVITY_TEST: Rh = 0
#             if '8' in SENSITIVITY_TEST: Rb = 0
#         if 'b' in SENSITIVITY_TEST:
#             if '5' in SENSITIVITY_TEST: Ry = 10
#             if '6' in SENSITIVITY_TEST: Ru = 100
#             if '7' in SENSITIVITY_TEST: Rh = 10
#             if '8' in SENSITIVITY_TEST: Rb = 10
#         if 'c' in SENSITIVITY_TEST:
#             if '5' in SENSITIVITY_TEST: Ry = 100
#             if '6' in SENSITIVITY_TEST: Ru = 1000
#             if '7' in SENSITIVITY_TEST: Rh = 100
#             if '8' in SENSITIVITY_TEST: Rb = 100

#         # Soft constraint combining age + elev deviations
#         age_factor = (ages - ages.min()) / (ages.max() - ages.min())
#         elevation_diff = elevations - mu
#         age_scaled = age_factor / age_uncertainties
#         elev_scaled = elevation_diff / elevation_uncertainties
#         soft_penalty = -np.sum(np.maximum(0, elev_scaled + age_scaled)**2) * Ry

#         # Penalize non-monotonic spline coefficients
#         non_decreasing_penalty = -np.sum(np.maximum(0, coeffs[:-1] - coeffs[1:])**2) * Ru

#         # Highest-elevation consistency
#         highest_sample_age = ages[np.argmax(elevations)]
#         idx = np.where(ages == highest_sample_age)[0][0]
#         highest_elevation_penalty = -((mu[idx] - elevations.max())**2) * Rh

#         # Youngest-age / lowest-elevation consistency
#         lowest_elevation_penalty = -((mu[0] - elevations.min())**2) * Rb

#         return ll + soft_penalty + non_decreasing_penalty + highest_elevation_penalty + lowest_elevation_penalty

#     # Proposal scale by test
#     proposal_scale = 60
#     if SENSITIVITY_TEST == 'C10a': proposal_scale = 10
#     if SENSITIVITY_TEST == 'C10b': proposal_scale = 100
#     if SENSITIVITY_TEST == 'C10c': proposal_scale = 200

#     # Sampling loop
#     for i in range(n_samples + burn_in):
#         for j in range(len(coeffs)):
#             current = coeffs[j]
#             proposal = current + np.random.normal(0, proposal_scale)
#             coeffs[j] = proposal
#             pll = log_likelihood(coeffs)
#             coeffs[j] = current
#             cll = log_likelihood(coeffs)
#             if np.random.rand() < np.exp(pll - cll):
#                 coeffs[j] = proposal
#         if i >= burn_in:
#             coeff_samples[i - burn_in] = coeffs

#     # Posterior summaries & high-res grid
#     coeff_mean  = coeff_samples.mean(axis=0)
#     coeff_lower = np.percentile(coeff_samples, 2.5, axis=0)
#     coeff_upper = np.percentile(coeff_samples, 97.5, axis=0)

#     ages_hr = np.arange(ages.min() - xstep, ages.max() + xstep, xstep)
#     B_hr = create_bspline_basis(ages_hr, knot_locs, degree)
#     spline_samples_hr = B_hr @ coeff_samples.T
#     spline_mean_hr = np.mean(spline_samples_hr, axis=1)
#     spline_lower_hr = np.percentile(spline_samples_hr, 2.5, axis=1)
#     spline_upper_hr = np.percentile(spline_samples_hr, 97.5, axis=1)

#     # Compute derivative samples & summaries
#     def create_bspline_basis_derivative(x, knots, degree):
#         extended = np.concatenate(([knots[0]]*degree, knots, [knots[-1]]*degree))
#         n_bases = len(knots) + degree - 1
#         Bp = np.zeros((len(x), n_bases))
#         for i in range(n_bases):
#             coeff = np.zeros(n_bases); coeff[i] = 1
#             Bp[:, i] = BSpline(extended, coeff, degree).derivative()(x)
#         return Bp

#     Bp_hr = create_bspline_basis_derivative(ages_hr, knot_locs, degree)
#     rate_samples_hr = Bp_hr @ coeff_samples.T
#     rate_mean_hr = rate_samples_hr.mean(axis=1)
#     rate_lower_hr = np.percentile(rate_samples_hr, 2.5, axis=1)
#     rate_upper_hr = np.percentile(rate_samples_hr, 97.5, axis=1)

#     return (
#         ages_hr,
#         spline_mean_hr, spline_lower_hr, spline_upper_hr, spline_samples_hr,
#         rate_mean_hr, rate_upper_hr, rate_lower_hr, rate_samples_hr
#     )


In [21]:
# import os
# import numpy as np
# import pandas as pd

# # your existing imports
# # from your_module import load_data, make_data_xy, run_bayesian_spline, N_POSTERIOR_SAMPLES

# # 1) All sensitivity tests
# test_configs = [ 'C3a']
#     # 'C1a','C1b','C1c',
#     # 'C3a','C3b','C3c',
#     # 'C4a','C4b','C4c',
#     # 'C5a','C5b','C5c',
#     # 'C6a','C6b','C6c',
#     # 'C7a','C7b','C7c',
#     # 'C8a','C8b','C8c',
#     # 'C10a','C10b','C10c',
#     # 'C11',
#     # 'C4aC6a','C4cC6a','C4aC6c','C4cC6c',
#     # 'C5cC6c','C5cC6cC7c','C5cC6cC7cC8c',
#     # 'C3cC11','C3cC11C4bC6b',
#     # 'C3cC11C4cC6b','C3cC11C4bC6bC7b',
#     # 'C3cC11C4bC10a','C3cC11C4bC10aC5bC6bC7bC8b',
#     # 'C3cC11C4cC10aC5bC6bC7bC8b',
#     # 'C3cC4cC5cC6c','C3cC11C5bC6b',
#     # 'C3cC11C10aC5bC6b','C11C4bC10aC5b',
#     # 'C3cC11C10aC5bC6bC7b','C4bC10aC5bC6bC7bC8b'


# # 2) The sites you want to run
# site_list = [
#     3,4,7,11,15,17,21,22,24,27,42,28,72,30,33,34,75,37,39,40,45,46,47,78,79,80,
#     48,49,50,52,53,54,55,56,57,59,60,61,62,63,64,66,67,68,70,85,87,88,90,91,95,
#     97,111,103,114,105,116,144,121,122,123,142,124,125,128,129,135,145,148,149,
#     151,152,156,163,164,165,166,168,169,161
# ]

# all_results = []

# for target_group in site_list:
#     results = []
#     for test in test_configs:
#         _, data, _ = load_data("icedcurrent_v14_EM.xlsx", SENSITIVITY_TEST=test)
#         grp = data[data["Group"] == target_group]
#         if grp.empty:
#             continue

#         # zero-age logic if only one point
#         if len(grp) == 1:
#             zeroage = grp['Best guess at when thickness reaches present'].iloc[0] * -1 - 20
#             new = grp.iloc[0].copy()
#             new.update({
#                 'elv_comp': 0, 'elv_err': 0.1,
#                 'age': zeroage, 'age_err': 0.1, 'name': 'zeropoint'
#             })
#             grp = pd.concat([grp, pd.DataFrame([new])], ignore_index=True)

#         # prepare data
#         x, y, xerrs, yerrs = make_data_xy(grp)

#         # run Bayesian spline
#         out = run_bayesian_spline(
#             x, xerrs, y, yerrs,
#             n_samples=(
#                 500  if test == 'C10a' else
#                 2000 if test == 'C10b' else
#                 3000 if test == 'C10c' else
#                 N_POSTERIOR_SAMPLES
#             ),
#             SENSITIVITY_TEST=test
#         )
#         x_fine, y_fine, *_, bsamps = out[:5]

#         # extend to full time
#         fullmin  = min(0, x_fine.min())
#         fulltime = np.arange(fullmin, 20000.0001, 5)
#         lo = fulltime[fulltime < x_fine.min()]
#         hi = fulltime[fulltime > x_fine.max()]
#         if len(lo) and round(lo[-1],4) == x_fine[0]:
#             lo = lo[:-1]
#         if len(hi) and round(hi[0],4) == x_fine[-1]:
#             hi = hi[1:]
#         loz = np.zeros(len(lo)) * np.nan
#         hiz = np.zeros(len(hi)) * np.nan
#         if y_fine.min() > 0:
#             loz = np.linspace(0, y_fine.min(), len(lo))
#         if 'YES' in grp['upper bound LGM ice extent?'].mode()[0]:
#             LGM = grp['rel_elv_LGM_m'].mean()
#             if LGM > 0:
#                 hiz = np.linspace(y_fine.max(), LGM, max(len(hi),2))
#         x_full = np.concatenate([lo, x_fine, hi])
#         y_full = np.concatenate([loz, y_fine, hiz])

#         # get predictions
#         if bsamps.ndim == 2:
#             n_samps = bsamps.shape[1]
#             # lo-bootstrap
#             boot_lo = np.tile(loz[:, None], (1, n_samps)) if len(lo) else np.zeros((0, n_samps))
#             # hi-bootstrap
#             boot_hi = np.tile(hiz[:, None], (1, n_samps)) if len(hi) else np.zeros((0, n_samps))
#             # concatenate and predict
#             bfull = np.concatenate([boot_lo, bsamps, boot_hi], axis=0)
#             y_pred = np.interp(x, x_full, np.nanmean(bfull, axis=1))
#         else:
#             y_pred = np.interp(x, x_full, y_full)

#         # compute WRSS
#         dy_dx     = np.gradient(y_pred, x)
#         total_var = yerrs**2 + (dy_dx * xerrs)**2
#         total_var[total_var == 0] = 1e-10
#         wrss      = np.sum(((y - y_pred)**2) / total_var)

#         # compute R²
#         ss_res = np.sum((y - y_pred)**2)
#         ss_tot = np.sum((y - np.mean(y))**2)
#         r2      = 1 - ss_res/ss_tot

#         results.append({
#             "Group":           target_group,
#             "SensitivityTest": test,
#             "WRSS":            wrss,
#             "R2":              r2
#         })

#     # pick the best test for this group
#     if results:
#         df = pd.DataFrame(results)
#         best = df.loc[df["R2"].idxmax()]
#         all_results.append(best.to_dict())

# # write all best-per-site into one Excel file
# output_file = "BestSensitivityTest/besttestssummaryC3a.xlsx"
# os.makedirs(os.path.dirname(output_file), exist_ok=True)
# pd.DataFrame(all_results).sort_values("Group") \
#     .to_excel(output_file, index=False)

# print(f"Wrote best-test summary for {len(all_results)} sites to:\n  {output_file}")


In [ ]:
# for site in ds_sites.site[:10]:
#    ds_sites.elv_post_samples.chunk('auto').mean('samples').sel(site=site).plot()

In [ ]:
# for sample in range(100):
#     ds_sites.elv_post_samples.chunk('auto').sel(site=121).sel(samples=sample).plot(alpha=0.2)